# claudeknows: try the spatial model on your photos (quick)

Loads a detector you already trained with `spatial_colab.ipynb` (no training here), then you
upload photos and see boxes, labels and distances drawn on them. Takes a few minutes, mostly
installing and downloading the depth model.

**Before you start:** Runtime -> Change runtime type -> T4 GPU (it also works without a GPU,
just slower).

## 1. Get the code

In [ ]:
REPO = "https://github.com/alexg-sketch/claudeknows"
BRANCH = "main"
import os
if not os.path.isdir("claudeknows_repo"):
    !git clone -b $BRANCH $REPO claudeknows_repo
%cd claudeknows_repo
!pip install -q -r requirements.txt transformers

## 2. Find your trained model

Looks in Google Drive first (`claudeknows_models/detector.pt`, saved there by the spatial
notebook). If it is not there, a button asks you to upload `detector.pt`.

In [ ]:
CHECKPOINT = "/content/drive/MyDrive/claudeknows_models/detector.pt"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive not connected:", e)
if not os.path.exists(CHECKPOINT):
    from google.colab import files
    print("No detector.pt in Drive. Upload the detector.pt from your spatial notebook run:")
    up = files.upload()
    CHECKPOINT = next((n for n in up if n.endswith(".pt")), None)
    assert CHECKPOINT, "No .pt file uploaded."
print("Using", CHECKPOINT)

## 3. Load the detector and the depth models

In [ ]:
import torch
from claudeknows.detector import load_detector, detect
from claudeknows.distance import (depth_models, estimate_distances, focal_px_from_exif,
                                  describe, draw_results)
device = "cuda" if torch.cuda.is_available() else "cpu"
model, classes = load_detector(CHECKPOINT, device)
relative_fn, metric_fn = depth_models()
print("Ready. Objects it knows:", ", ".join(classes))

## 4. Upload photos

Run this cell again as many times as you like. Photos straight from your phone work best (their
camera info helps the distance estimate). Near/far order is reliable; metres are rough.

In [ ]:
from PIL import Image
from google.colab import files
up = files.upload()
for name in up:
    image = Image.open(name)
    focal = focal_px_from_exif(image)  # None without camera info: then only the depth model is used
    image = image.convert("RGB")
    results = estimate_distances(image, detect(model, image, classes), relative_fn, metric_fn, focal)
    print(name); print("\n".join(describe(results)) or "Nothing found.")
    display(draw_results(image, results))